# Step 5c — Why do our counts and the published counts disagree for some anime?

**Background.** In step 5a we compared, for every anime, the published `members` and `scored_by` with our own counts from the list table. The order agrees almost perfectly, but two things are unexplained:

- Our count is **larger** than the published one for **2 anime** (`members`) and **59 anime** (`scored_by`, up to 64 times). If our users were only a part of the published ones, this could not happen.
- The ratio (ours ÷ published) is far from constant: the middle 90% of anime range from 0.21 to 0.55 for `members`.

**Three open questions, one part each.**

| Part | Question | What would make us worry |
|---|---|---|
| 1 | Do `members` and `scored_by` contain missing, zero or negative values? | Anime that were silently dropped from the comparison, or ratios that divide by zero |
| 2 | Were the anime table and the list table collected at about the same time? | A big gap in time would explain why our counts can be larger than the published ones |
| 3 | Are the anime with a very high or very low ratio errors, or is there a pattern? | A pattern (by type, by status, by year) means a bias in the sample. Random oddities mean data errors |

**Important limit on part 2.** Neither table has a column that says when it was crawled. We can only use **proxies**: the latest `my_last_updated` in the list table, the latest `last_online` in the user table, and the start date of each anime. A proxy gives a bound, not the crawl date.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. It reads 3 columns of the 80M-row list table (a minute or two).

**Where this step sits.** Step 5 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). It compares our counts with the published counts, so a cleaning rule that drops rows changes its numbers. Run it on the cleaned data of step 4 before reporting a number from here.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 40)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

anime = pd.read_csv(paths["AnimeList.csv"], usecols=[
    "anime_id", "title", "type", "status", "aired_string", "members", "scored_by", "score", "favorites"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], usecols=["username", "last_online"], **na_kwargs)
lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=["anime_id", "my_score", "my_last_updated"],
                    dtype={"anime_id": "int32", "my_score": "int8", "my_last_updated": "int64"}, **na_kwargs)
print(f"anime: {len(anime):,} | users: {len(users):,} | list rows: {len(lists):,}")

## 1. Missing, zero or negative values in `members` and `scored_by`

A ratio ours ÷ published only exists when the published value is above 0. The cell below counts every kind of unusable value, then lists the anime that stay out of the comparison.

In [ ]:
for col in ["members", "scored_by"]:
    s = anime[col]
    print(f"{col:10s} missing: {int(s.isna().sum()):>5,} | zero: {int((s == 0).sum()):>5,} | negative: {int((s < 0).sum()):>5,}"
          f" | min: {s.min():,.0f} | median: {s.median():,.0f} | max: {s.max():,.0f}")

print("\nscored_by larger than members:", int((anime["scored_by"] > anime["members"]).sum()), "anime")

# Anime that cannot be compared (published value is 0 or less, or missing)
unusable = anime[~(anime["members"] > 0) | ~(anime["scored_by"] > 0)]
print(f"\nanime with members <= 0 or scored_by <= 0: {len(unusable):,}")
print("by type  :", unusable["type"].value_counts().to_dict())
print("by status:", unusable["status"].value_counts().to_dict())
print("\nmean of `score` for them (0 or empty would mean 'no score yet'):",
      round(unusable["score"].mean(), 2))
print(unusable[["anime_id", "title", "type", "status", "members", "scored_by", "score"]].head(10).to_string(index=False))

### Observations for part 1

**TODO (fill in after the run):** Are there missing, zero or negative values? How many anime are left out of the `scored_by` comparison because `scored_by` is 0, and what are they (type, status, score)? Do the anime with `scored_by = 0` have no score, which would make sense?

## 2. Were the two tables collected around the same time?

If the list table was crawled **after** the anime table, users could have added anime in between, and our count could be larger than the published one. We look for the time range of the list table and compare it with the anime's own dates.

- `my_last_updated` is a Unix timestamp in seconds: the last time a user changed that row. Its **latest values** show when the lists were crawled, at the earliest.
- `last_online` in the user table is the last time the user was seen online, so its latest value is a second proxy for the user crawl.
- The anime table has no crawl date. An anime that **started airing after** the end of the lists is a clue that the anime table was crawled later.

In [ ]:
ts = lists.loc[lists["my_last_updated"] > 0, "my_last_updated"]
print(f"rows with my_last_updated <= 0: {int((lists['my_last_updated'] <= 0).sum()):,} (placeholder, left out below)")
quant = ts.quantile([0, .001, .01, .5, .99, .999, 1])
print("\nmy_last_updated, as dates:")
for q, v in quant.items():
    print(f"  {q:>6.1%}  {pd.to_datetime(v, unit='s')}")

online = pd.to_datetime(users["last_online"], errors="coerce")
print(f"\nlast_online: empty or unreadable {int(online.isna().sum()):,} | before 1990 (placeholder) {int((online.dt.year < 1990).sum()):,}")
real_online = online[online.dt.year >= 1990]
print("  real values: earliest", real_online.min(), "| 99.9th percentile", real_online.quantile(.999), "| latest", real_online.max())

# Anime start date, from the text "Apr 3, 2002 to Sep 18, 2002"
start = pd.to_datetime(anime["aired_string"].str.split(" to ").str[0], errors="coerce", format="mixed")
anime["start"] = start
print(f"\nanime with a readable start date: {int(start.notna().sum()):,} of {len(anime):,}")
print("start dates: earliest", start.min(), "| latest", start.max())
lists_end = pd.to_datetime(ts.quantile(.999), unit="s")
after = anime[anime["start"] > lists_end]
print(f"\nanime that started airing after {lists_end.date()} (99.9th percentile of my_last_updated): {len(after):,}")
print("  with members > 0:", int((after["members"] > 0).sum()), "| median members:", after["members"].median())

### Observations for part 2

**TODO (fill in after the run):** When do the lists end (latest `my_last_updated`) and when do users end (`last_online`)? Is there a long gap before the latest anime start dates? How many anime started after the lists ended, and do they still have members? Write the gap in months or years. If there are anime that started airing after the lists ended, the two tables were crawled at different times.

## 3. Which anime have a very high or very low ratio?

For every anime we compute our count ÷ the published one. If the extremes come from a **pattern** (a type, a status, a period), the sample is biased in that direction. If they look random, they are more likely data errors. We list the extremes and then look at the median ratio by group.

In [ ]:
ours = pd.DataFrame({
    "on_list": lists.groupby("anime_id").size(),
    "rated": lists[lists["my_score"] > 0].groupby("anime_id").size(),
})
a = anime.set_index("anime_id").join(ours, how="left").fillna({"on_list": 0, "rated": 0})
a["ratio_members"] = a["on_list"] / a["members"].where(a["members"] > 0)
a["ratio_scored"] = a["rated"] / a["scored_by"].where(a["scored_by"] > 0)
a["start_year"] = a["start"].dt.year

cols = ["title", "type", "status", "start_year", "members", "on_list", "ratio_members"]
print("highest ratio for members (ours / published):")
print(a.nlargest(10, "ratio_members")[cols].round(3).to_string())
print("\nlowest ratio for members (only anime with at least 1 of our users):")
print(a[a["on_list"] > 0].nsmallest(10, "ratio_members")[cols].round(3).to_string())

over = a[a["ratio_scored"] > 1].sort_values("ratio_scored", ascending=False)
print(f"\nanime where our rated count is larger than scored_by: {len(over)}")
print(over[["title", "type", "status", "start_year", "members", "scored_by", "rated", "ratio_scored", "score"]].head(15).round(2).to_string())

In [ ]:
# Is the ratio a pattern? Median ratio for members by group (groups with at least 30 anime)
def by_group(key, label):
    g = a.groupby(key, observed=True)["ratio_members"].agg(anime="size", median="median",
                                                           p10=lambda s: s.quantile(.1), p90=lambda s: s.quantile(.9))
    print(f"\nmedian ratio by {label}:")
    print(g[g["anime"] >= 30].round(3).to_string())

by_group("type", "type")
by_group("status", "status")
a["start_period"] = pd.cut(a["start_year"], [0, 1990, 2000, 2005, 2010, 2013, 2020], right=False)
by_group("start_period", "start year")

# Small anime: is the ratio more extreme when the published number is small?
a["members_group"] = pd.cut(a["members"], [0, 100, 1000, 10000, 100000, np.inf])
by_group("members_group", "size of `members`")

### Observations for part 3

**TODO (fill in after the run):** What do the extremes have in common (type, status, year, size)? Does the median ratio change with the start year, with the type, or with the size of the anime? Do the 59 anime with `rated > scored_by` share something? Say whether the sample is biased in a direction (a pattern) or whether the extremes look like isolated data errors.

## Reading the result

| What we see | What it means |
|---|---|
| Median ratio changes strongly with start year or type | The crawled users are not a uniform slice of MAL: they favour some periods or types. Results about popularity must say so |
| The anime with `rated > scored_by` are mostly small (few members) or new | The published `scored_by` was probably counted at a different moment, or is hidden for little-rated anime. We keep using `scored_by` for order only |
| Lists end long before the latest anime start dates | The two tables were crawled at different times, which can make our count larger for some anime |
| No pattern, extremes scattered | Isolated data errors. We list them and exclude them from any ratio-based claim |